# Gemma 4 MAX vs vLLM Colab Benchmark

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrrahman1517/nemotron-vl-embed-stack/blob/codex-gemma4-max-vs-vllm-colab/gemma4_max_vs_vllm_colab.ipynb)

This notebook is designed to test the April 2, 2026 Modular blog claim that **MAX delivers 15% higher throughput than vLLM for Gemma 4 on NVIDIA B200**.

What this notebook does:

- runs **MAX** and **vLLM** side by side against the same selected model on the current runtime
- benchmarks both with the **same harness**: `max benchmark`
- reports request throughput, output token throughput, TTFT, TPOT, ITL, and optional GPU stats

Important caveat:

- the blog claim is about **NVIDIA B200** hardware
- most Colab runtimes provide **L4** or **A100**, not B200
- if you run this on L4 or A100, the result is still useful, but it is a **directional comparison**, not an exact reproduction of the published number
- if you run this on a **T4 or other sub-24GB GPU**, the notebook now drops into a **small-GPU fallback mode** with a much smaller model so you can still compare MAX and vLLM on the same machine


## Source Notes

Primary sources used to shape this notebook:

- Modular blog claim and benchmark footnote: [Day Zero Launch: Fastest Performance for Gemma 4 on NVIDIA and AMD](https://www.modular.com/blog/day-zero-launch-fastest-performance-for-gemma-4-on-nvidia-and-amd)
- MAX package installation and GPU compatibility: [Modular packages](https://docs.modular.com/stable/max/packages/)
- MAX supported-model guidance and Gemma 4 architecture support: [Supported models](https://docs.modular.com/max/models/)
- MAX benchmark CLI, including `modular-chat` and `vllm` backends: [max benchmark](https://docs.modular.com/max/cli/benchmark/)
- MAX CLI serve example: [max CLI](https://docs.modular.com/stable/max/max-cli)
- vLLM Gemma 4 deployment guidance and minimum GPU sizes: [Gemma 4 Usage Guide - vLLM](https://docs.vllm.ai/projects/recipes/en/latest/Google/Gemma4.html)

Practical interpretation:

- if the runtime has about **24 GiB** VRAM, this notebook defaults to `google/gemma-4-E4B-it`
- if the runtime has about **80 GiB** VRAM, it defaults to `google/gemma-4-26B-A4B-it`
- if the runtime has about **14 to 16 GiB** VRAM, it falls back to `allenai/OLMo-2-0425-1B-Instruct`
- only a **B200-class** run can directly test the blog's exact hardware claim


## Recommended Runtime Settings

Use these as starting points before you run the benchmark:

- **Colab L4 (24 GiB)**: keep the default `google/gemma-4-E4B-it`, use `NUM_PROMPTS = 64`, `MAX_CONCURRENCY = 8`, and leave `GPU_MEMORY_UTILIZATION = 0.90`
- **Colab A100 (40 GiB)**: still use `google/gemma-4-E4B-it`, and you can usually try `NUM_PROMPTS = 96` to `128` and `MAX_CONCURRENCY = 8` to `16`
- **80 GiB GPU or larger**: switch to `google/gemma-4-26B-A4B-it`, use `NUM_PROMPTS = 128`, `MAX_CONCURRENCY = 16`, and keep `GPU_MEMORY_UTILIZATION` between `0.85` and `0.90`
- **T4 / 16 GiB-class GPU**: let the notebook fall back to `allenai/OLMo-2-0425-1B-Instruct`, use shorter contexts, and treat the result as a methodology check only
- **B200**: this is the only class of hardware that can directly test the published claim instead of just giving a directional comparison

If either engine OOMs during load, lower `GPU_MEMORY_UTILIZATION` first, then reduce `MAX_CONCURRENCY`, and finally reduce `NUM_PROMPTS`.


## Before You Run

You may need Hugging Face access for the model the notebook will use.

Recommended setup:

- accept the Hugging Face model license for the model you plan to test if it is gated
- create a Hugging Face access token with model download access
- store it in a Colab secret named `HF_TOKEN` when you are using a gated model such as Gemma 4

If you do not add the secret ahead of time, the notebook will prompt you for the token with hidden input only when the selected model likely needs it.


## MAX-Specific Reality Check

If the notebook appears to hang during `max serve`, the usual causes are:

- the first MAX run is still downloading weights or compiling kernels
- the current runtime is using an NVIDIA driver older than what current MAX docs require
- the selected model is too large for the available runtime and batch settings

This notebook now performs a driver preflight and a `max warm-cache` step before starting the server so those issues are easier to distinguish.


In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "uv", "requests", "pandas", "matplotlib"],
    check=True,
)


In [ ]:
%%writefile gemma4_colab_benchmark_helper.py
from __future__ import annotations

import json
import os
import shlex
import signal
import subprocess
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

try:
    import requests
except ImportError:  # pragma: no cover - installed inside Colab
    requests = None


@dataclass
class GPUInfo:
    name: str
    memory_gb: float
    driver_version: str
    compute_capability: str | None = None


@dataclass
class ManagedProcess:
    name: str
    command: list[str]
    process: subprocess.Popen[str]
    log_path: Path


def run(cmd: list[str], *, env: dict[str, str] | None = None, cwd: str | None = None) -> None:
    print("$", shlex.join(cmd))
    subprocess.run(cmd, check=True, env=env, cwd=cwd)


def run_best_effort(
    cmd: list[str],
    *,
    env: dict[str, str] | None = None,
    cwd: str | None = None,
) -> bool:
    print("$", shlex.join(cmd))
    completed = subprocess.run(
        cmd,
        check=False,
        text=True,
        capture_output=True,
        env=env,
        cwd=cwd,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    return completed.returncode == 0


def capture(cmd: list[str], *, env: dict[str, str] | None = None, cwd: str | None = None) -> str:
    print("$", shlex.join(cmd))
    completed = subprocess.run(
        cmd,
        check=True,
        text=True,
        capture_output=True,
        env=env,
        cwd=cwd,
    )
    return completed.stdout.strip()


def detect_gpu() -> GPUInfo | None:
    try:
        output = capture(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total,driver_version,compute_cap",
                "--format=csv,noheader,nounits",
            ]
        )
    except Exception:
        return None

    first_line = output.splitlines()[0].strip()
    parts = [part.strip() for part in first_line.split(",")]
    if len(parts) < 3:
        return None

    name = parts[0]
    memory_gb = round(float(parts[1]) / 1024.0, 2)
    driver_version = parts[2]
    compute_capability = parts[3] if len(parts) >= 4 else None
    return GPUInfo(
        name=name,
        memory_gb=memory_gb,
        driver_version=driver_version,
        compute_capability=compute_capability,
    )


def choose_benchmark_model(gpu: GPUInfo | None, force_model: str | None = None) -> tuple[str, str]:
    if force_model:
        return force_model, "Using a user-forced model override."

    if gpu is None:
        return (
            "allenai/OLMo-2-0425-1B-Instruct",
            "GPU detection failed; defaulting to a small open compatibility model so the notebook can still validate the benchmark flow.",
        )

    if gpu.memory_gb >= 75:
        if "B200" in gpu.name.upper():
            return (
                "google/gemma-4-26B-A4B-it",
                "This runtime is closest to the blog's hardware class, though exact parity still depends on the full software stack.",
            )
        return (
            "google/gemma-4-26B-A4B-it",
            "This runtime has enough memory for the larger Gemma 4 MoE model, but it is still not the exact B200 setup from the blog.",
        )

    if gpu.memory_gb >= 22:
        return (
            "google/gemma-4-E4B-it",
            "This is a directional Colab-friendly run on a smaller Gemma 4 variant because typical Colab GPUs do not expose B200-class memory.",
        )

    if gpu.memory_gb >= 14:
        return (
            "allenai/OLMo-2-0425-1B-Instruct",
            "No Gemma 4 model cleanly fits on this GPU. Falling back to a smaller open OLMo model so you can still compare MAX and vLLM on the same runtime. Treat this as a methodology check, not verification of the Gemma 4 claim.",
        )

    raise RuntimeError(
        f"Detected only {gpu.memory_gb:.1f} GiB of VRAM on {gpu.name}. "
        "This is too small for the Gemma 4 models targeted by this notebook and likely too tight even for a useful fallback benchmark."
    )


def start_logged_process(
    name: str,
    command: list[str],
    *,
    log_dir: str | Path,
    env: dict[str, str] | None = None,
    cwd: str | None = None,
) -> ManagedProcess:
    log_dir = Path(log_dir)
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f"{name}.log"
    log_file = log_path.open("w", encoding="utf-8")

    print("$", shlex.join(command))
    process = subprocess.Popen(
        command,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
        cwd=cwd,
    )
    return ManagedProcess(name=name, command=command, process=process, log_path=log_path)


def tail_log(log_path: str | Path, lines: int = 80) -> str:
    log_path = Path(log_path)
    if not log_path.exists():
        return f"{log_path} does not exist."
    content = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
    return "\n".join(content[-lines:])


def ensure_server_ready(base_url: str, *, timeout_s: int = 1800) -> None:
    if requests is None:
        raise RuntimeError("requests is required for readiness checks.")

    deadline = time.time() + timeout_s
    last_error = None
    while time.time() < deadline:
        try:
            response = requests.get(f"{base_url.rstrip('/')}/v1/models", timeout=10)
            if response.ok:
                print(f"Server ready at {base_url}")
                return
            last_error = f"HTTP {response.status_code}: {response.text[:200]}"
        except Exception as exc:  # pragma: no cover - runtime specific
            last_error = str(exc)
        time.sleep(5)
    raise TimeoutError(f"Server at {base_url} did not become ready within {timeout_s}s. Last error: {last_error}")


def ensure_server_ready_with_logs(
    base_url: str,
    handle: ManagedProcess,
    *,
    timeout_s: int = 1800,
    log_interval_s: int = 30,
) -> None:
    if requests is None:
        raise RuntimeError("requests is required for readiness checks.")

    deadline = time.time() + timeout_s
    last_error = None
    next_log_time = time.time() + log_interval_s

    while time.time() < deadline:
        if handle.process.poll() is not None:
            raise RuntimeError(
                f"{handle.name} exited before becoming ready. "
                f"Last log lines:\n{tail_log(handle.log_path, lines=120)}"
            )

        try:
            response = requests.get(f"{base_url.rstrip('/')}/v1/models", timeout=10)
            if response.ok:
                print(f"Server ready at {base_url}")
                return
            last_error = f"HTTP {response.status_code}: {response.text[:200]}"
        except Exception as exc:  # pragma: no cover - runtime specific
            last_error = str(exc)

        if time.time() >= next_log_time:
            print(f"Still waiting for {handle.name} at {base_url}")
            print(tail_log(handle.log_path, lines=80))
            next_log_time = time.time() + log_interval_s

        time.sleep(5)

    raise TimeoutError(
        f"Server at {base_url} did not become ready within {timeout_s}s. "
        f"Last error: {last_error}\nLast log lines:\n{tail_log(handle.log_path, lines=120)}"
    )


def warmup_chat(base_url: str, model: str) -> dict[str, Any]:
    if requests is None:
        raise RuntimeError("requests is required for warmup requests.")

    payload = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": "Reply with the single word ready.",
            }
        ],
        "temperature": 0,
        "max_tokens": 8,
    }
    response = requests.post(
        f"{base_url.rstrip('/')}/v1/chat/completions",
        json=payload,
        timeout=300,
    )
    response.raise_for_status()
    body = response.json()
    print(json.dumps(body.get("usage", {}), indent=2))
    return body


def stop_process(handle: ManagedProcess | None, *, grace_seconds: int = 20) -> None:
    if handle is None:
        return
    process = handle.process
    if process.poll() is not None:
        return

    print(f"Stopping {handle.name} (pid={process.pid})")
    process.send_signal(signal.SIGINT if sys.platform != "win32" else signal.CTRL_BREAK_EVENT)
    try:
        process.wait(timeout=grace_seconds)
    except subprocess.TimeoutExpired:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=10)


def run_benchmark(
    *,
    max_bin: str,
    backend: str,
    model: str,
    port: int,
    result_dir: str | Path,
    result_filename: str,
    dataset_name: str = "random",
    num_prompts: int = 128,
    request_rate: str = "inf",
    max_concurrency: int = 16,
    random_input_len: int = 550,
    random_output_len: int = 256,
    random_coefficient_of_variation: str = "0.0,0.0",
    collect_gpu_stats: bool = True,
    tokenizer: str | None = None,
    env: dict[str, str] | None = None,
) -> Path:
    result_dir = Path(result_dir)
    result_dir.mkdir(parents=True, exist_ok=True)
    result_path = result_dir / result_filename
    if result_path.exists():
        result_path.unlink()

    command = [
        max_bin,
        "benchmark",
        "--model",
        model,
        "--backend",
        backend,
        "--endpoint",
        "/v1/chat/completions",
        "--host",
        "127.0.0.1",
        "--port",
        str(port),
        "--dataset-name",
        dataset_name,
        "--num-prompts",
        str(num_prompts),
        "--request-rate",
        str(request_rate),
        "--max-concurrency",
        str(max_concurrency),
        "--result-dir",
        str(result_dir),
        "--result-filename",
        result_filename,
    ]

    if tokenizer:
        command.extend(["--tokenizer", tokenizer])

    if collect_gpu_stats:
        command.append("--collect-gpu-stats")

    if dataset_name == "random":
        command.extend(
            [
                "--random-input-len",
                str(random_input_len),
                "--random-output-len",
                str(random_output_len),
                "--random-coefficient-of-variation",
                random_coefficient_of_variation,
            ]
        )

    save_flags = ("--save-results", "--save-result")
    last_error: Exception | None = None
    for save_flag in save_flags:
        cmd = [*command, save_flag]
        try:
            run(cmd, env=env)
            return result_path
        except subprocess.CalledProcessError as exc:
            last_error = exc
            if result_path.exists():
                return result_path
            print(f"{save_flag} failed, retrying with the alternate save flag.")

    if result_path.exists():
        return result_path
    raise RuntimeError(f"Benchmark did not produce {result_path}") from last_error


def load_benchmark_result(result_path: str | Path) -> dict[str, Any]:
    result_path = Path(result_path)
    return json.loads(result_path.read_text(encoding="utf-8"))


def summarize_result(result: dict[str, Any], label: str) -> dict[str, Any]:
    summary = {
        "label": label,
        "backend": result.get("backend"),
        "model_id": result.get("model_id"),
        "completed": result.get("completed"),
        "duration_s": result.get("duration"),
        "request_throughput": result.get("request_throughput"),
        "output_throughput": result.get("output_throughput"),
        "total_token_throughput": result.get("total_token_throughput"),
        "mean_ttft_ms": result.get("mean_ttft_ms"),
        "p99_ttft_ms": result.get("p99_ttft_ms"),
        "mean_tpot_ms": result.get("mean_tpot_ms"),
        "mean_itl_ms": result.get("mean_itl_ms"),
        "max_concurrency": result.get("max_concurrency"),
        "gpu_utilization": result.get("gpu_utilization"),
        "peak_gpu_memory_used": result.get("peak_gpu_memory_used"),
    }
    return summary


def percent_delta(numerator: float | int | None, denominator: float | int | None) -> float | None:
    if numerator is None or denominator in (None, 0):
        return None
    return ((float(numerator) - float(denominator)) / float(denominator)) * 100.0


In [ ]:
import getpass
import os
import shutil
from pathlib import Path

from gemma4_colab_benchmark_helper import choose_benchmark_model, detect_gpu

os.environ.setdefault("HF_HOME", "/content/hf-cache")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

gpu = detect_gpu()
print("Detected GPU:", gpu)

FORCE_MODEL = None
MODEL, MODEL_NOTE = choose_benchmark_model(gpu, FORCE_MODEL)
IS_SMALL_GPU_FALLBACK = MODEL == "allenai/OLMo-2-0425-1B-Instruct"

HF_TOKEN = os.environ.get("HF_TOKEN", "")
try:
    from google.colab import userdata

    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
except Exception:
    pass

MODEL_LIKELY_GATED = MODEL.startswith(("google/", "meta-llama/"))

if MODEL_LIKELY_GATED and not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Enter your Hugging Face token (input hidden): ").strip()

MODULAR_CHANNEL = "stable"  # change to "nightly" if you want the newest MAX build
MODULAR_VERSION = "26.2"
VLLM_VERSION = "0.18.2rc1.dev7"
GPU_MEMORY_UTILIZATION = 0.90
FAIL_ON_UNSUPPORTED_MAX_DRIVER = True
RUN_MAX_WARM_CACHE = True
MAX_STARTUP_TIMEOUT_S = 1800
BENCHMARK_DATASET = "random"
RANDOM_COV = "0.0,0.0"
REQUEST_RATE = "inf"
MAX_PORT = 8000
VLLM_PORT = 8001
LOG_DIR = Path("/content/benchmark-logs")
RESULT_DIR = Path("/content/benchmark-results")
ENV_ROOT = Path("/content/benchmark-envs")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
elif MODEL_LIKELY_GATED:
    raise ValueError(
        "No Hugging Face token was provided for a gated model. Add a Colab secret named HF_TOKEN, "
        "set os.environ['HF_TOKEN'], or paste the token into the hidden prompt."
    )
else:
    print("Proceeding without HF_TOKEN because the selected fallback model is open.")

if IS_SMALL_GPU_FALLBACK:
    GPU_MEMORY_UTILIZATION = 0.85
    MAX_BATCH_SIZE = 4
    NUM_PROMPTS = 32
    MAX_CONCURRENCY = 4
    RANDOM_INPUT_LEN = 256
    RANDOM_OUTPUT_LEN = 96
    MAX_MODEL_LEN = 4096
else:
    MAX_BATCH_SIZE = 4 if gpu and gpu.memory_gb < 90 else 8
    NUM_PROMPTS = 64 if gpu and gpu.memory_gb < 75 else 128
    MAX_CONCURRENCY = 8 if gpu and gpu.memory_gb < 75 else 16
    RANDOM_INPUT_LEN = 550
    RANDOM_OUTPUT_LEN = 256
    MAX_MODEL_LEN = 8192 if "E4B" in MODEL else 32768

print("Selected model:", MODEL)
print("Selection note:", MODEL_NOTE)
if gpu and "B200" not in gpu.name.upper():
    print("This is not a B200 runtime, so the result should be interpreted as directional rather than exact.")
if IS_SMALL_GPU_FALLBACK:
    print("Small-GPU fallback mode is active. This run is for MAX-vs-vLLM methodology comparison only, not Gemma 4 claim verification.")
if gpu:
    driver_major = None
    try:
        driver_major = int(gpu.driver_version.split(".")[0])
    except Exception:
        print("Warning: could not parse the NVIDIA driver version from", gpu.driver_version)
    if (not IS_SMALL_GPU_FALLBACK) and driver_major is not None and driver_major < 580:
        ptxas_path = shutil.which("ptxas") or "/usr/local/cuda/bin/ptxas"
        if os.path.exists(ptxas_path):
            os.environ["MODULAR_NVPTX_COMPILER_PATH"] = ptxas_path
            print(
                "Detected an older NVIDIA driver, so MAX will use "
                f"MODULAR_NVPTX_COMPILER_PATH={ptxas_path} to try the documented bypass."
            )
        else:
            message = (
                f"Detected NVIDIA driver {gpu.driver_version}. Current MAX docs list driver 580+ "
                "for supported GPU acceleration. Colab often uses an older driver, which can make "
                "MAX startup hang or fail. A documented workaround is setting "
                "MODULAR_NVPTX_COMPILER_PATH to a system ptxas binary, but none was found in this "
                "runtime. Use a supported host for a reliable MAX run, or set "
                "FAIL_ON_UNSUPPORTED_MAX_DRIVER = False if you want to try anyway."
            )
            if FAIL_ON_UNSUPPORTED_MAX_DRIVER:
                raise RuntimeError(message)
            print("Warning:", message)


In [ ]:
import os
import subprocess
from pathlib import Path


def ensure_uv_env(env_path: Path) -> Path:
    if not env_path.exists():
        subprocess.run(["uv", "venv", str(env_path)], check=True)
    return env_path / "bin" / "python"


ENV_ROOT.mkdir(parents=True, exist_ok=True)
MAX_ENV = ENV_ROOT / "max"
VLLM_ENV = ENV_ROOT / "vllm"

max_python = ensure_uv_env(MAX_ENV)
vllm_python = ensure_uv_env(VLLM_ENV)

if MODULAR_CHANNEL == "nightly":
    subprocess.run(
        [
            "uv",
            "pip",
            "install",
            "--python",
            str(max_python),
            "modular",
            "--index",
            "https://whl.modular.com/nightly/simple/",
            "--prerelease",
            "allow",
        ],
        check=True,
    )
    ACTUAL_MODULAR_SPEC = "modular (nightly)"
else:
    subprocess.run(
        [
            "uv",
            "pip",
            "install",
            "--python",
            str(max_python),
            f"modular=={MODULAR_VERSION}",
            "--extra-index-url",
            "https://modular.gateway.scarf.sh/simple/",
        ],
        check=True,
    )
    ACTUAL_MODULAR_SPEC = f"modular=={MODULAR_VERSION}"

subprocess.run(
    ["uv", "pip", "install", "--python", str(max_python), "requests", "protobuf==6.33.5"],
    check=True,
)

ACTUAL_VLLM_SPEC = f"vllm=={VLLM_VERSION}"
try:
    subprocess.run(
        [
            "uv",
            "pip",
            "install",
            "--python",
            str(vllm_python),
            f"vllm=={VLLM_VERSION}",
            "--torch-backend=auto",
            "--extra-index-url",
            "https://wheels.vllm.ai/nightly",
            "--prerelease",
            "allow",
        ],
        check=True,
    )
except subprocess.CalledProcessError:
    print("Exact vLLM prerelease install failed; falling back to the latest available vLLM build.")
    subprocess.run(
        [
            "uv",
            "pip",
            "install",
            "--python",
            str(vllm_python),
            "vllm",
            "--torch-backend=auto",
        ],
        check=True,
    )
    ACTUAL_VLLM_SPEC = "vllm (latest available)"

subprocess.run(
    ["uv", "pip", "install", "--python", str(vllm_python), "transformers==5.5.0"],
    check=True,
)

MAX_BIN = str(MAX_ENV / "bin" / "max")
VLLM_BIN = str(VLLM_ENV / "bin" / "vllm")
MAX_SITE_PACKAGES = subprocess.run(
    [str(max_python), "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
VLLM_SITE_PACKAGES = subprocess.run(
    [str(vllm_python), "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

max_version = subprocess.run([MAX_BIN, "--version"], capture_output=True, text=True).stdout.strip()
vllm_version_output = subprocess.run([VLLM_BIN, "--version"], capture_output=True, text=True).stdout.strip()
max_protobuf_probe_code = '''
import importlib.metadata as md
import traceback

try:
    print("protobuf dist:", md.version("protobuf"))
except Exception as exc:
    print("protobuf dist lookup failed:", repr(exc))

try:
    from google.protobuf import runtime_version as rv
    print("runtime file:", getattr(rv, "__file__", "<none>"))
    print("runtime version attr:", getattr(rv, "PROTOBUF_RUNTIME_VERSION", None))
    print(
        "runtime tuple:",
        getattr(rv, "MAJOR", None),
        getattr(rv, "MINOR", None),
        getattr(rv, "PATCH", None),
        getattr(rv, "SUFFIX", None),
    )
except Exception:
    traceback.print_exc()
'''
max_protobuf_probe = subprocess.run(
    [str(max_python), "-c", max_protobuf_probe_code],
    capture_output=True,
    text=True,
    check=False,
)

print("MAX install:", ACTUAL_MODULAR_SPEC)
print("MAX version:", max_version)
print("MAX site-packages:", MAX_SITE_PACKAGES)
print("MAX protobuf probe exit:", max_protobuf_probe.returncode)
if max_protobuf_probe.stdout:
    print(max_protobuf_probe.stdout.strip())
if max_protobuf_probe.stderr:
    print(max_protobuf_probe.stderr.strip())
print("vLLM install:", ACTUAL_VLLM_SPEC)
print("vLLM version:", vllm_version_output)
print("vLLM site-packages:", VLLM_SITE_PACKAGES)


## Launch and Benchmark MAX

This uses the OpenAI-compatible `max serve` endpoint and then benchmarks it with `max benchmark --backend modular-chat`.


In [ ]:
from gemma4_colab_benchmark_helper import (
    ensure_server_ready_with_logs,
    run,
    run_best_effort,
    start_logged_process,
    tail_log,
    warmup_chat,
)

server_env = os.environ.copy()
if HF_TOKEN:
    server_env["HF_TOKEN"] = HF_TOKEN
server_env["HF_HOME"] = os.environ["HF_HOME"]
server_env["VIRTUAL_ENV"] = str(MAX_ENV)
server_env["PATH"] = f"{MAX_ENV / 'bin'}:{server_env['PATH']}"
server_env["PYTHONNOUSERSITE"] = "1"
server_env["PYTHONPATH"] = MAX_SITE_PACKAGES

if RUN_MAX_WARM_CACHE:
    warm_cache_ok = run_best_effort(
        [
            MAX_BIN,
            "warm-cache",
            "--model",
            MODEL,
        ],
        env=server_env,
    )
    if not warm_cache_ok:
        print("MAX warm-cache failed. Continuing to max serve anyway because warm-cache is optional.")

max_handle = start_logged_process(
    "max_server",
    [
        MAX_BIN,
        "serve",
        "--model",
        MODEL,
        "--devices",
        "gpu:0",
        "--port",
        str(MAX_PORT),
        "--max-length",
        str(MAX_MODEL_LEN),
        "--max-batch-size",
        str(MAX_BATCH_SIZE),
        "--device-memory-utilization",
        str(GPU_MEMORY_UTILIZATION),
    ],
    log_dir=LOG_DIR,
    env=server_env,
)

ensure_server_ready_with_logs(
    f"http://127.0.0.1:{MAX_PORT}",
    max_handle,
    timeout_s=MAX_STARTUP_TIMEOUT_S,
)
warmup_chat(f"http://127.0.0.1:{MAX_PORT}", MODEL)
print(tail_log(max_handle.log_path, lines=40))


In [ ]:
from gemma4_colab_benchmark_helper import load_benchmark_result, run_benchmark, summarize_result

max_result_path = run_benchmark(
    max_bin=MAX_BIN,
    backend="modular-chat",
    model=MODEL,
    port=MAX_PORT,
    result_dir=RESULT_DIR,
    result_filename="max_random.json",
    dataset_name=BENCHMARK_DATASET,
    num_prompts=NUM_PROMPTS,
    request_rate=REQUEST_RATE,
    max_concurrency=MAX_CONCURRENCY,
    random_input_len=RANDOM_INPUT_LEN,
    random_output_len=RANDOM_OUTPUT_LEN,
    random_coefficient_of_variation=RANDOM_COV,
    tokenizer=MODEL,
    env=server_env,
)

max_result = load_benchmark_result(max_result_path)
max_summary = summarize_result(max_result, "MAX")
max_summary


In [ ]:
from gemma4_colab_benchmark_helper import stop_process

stop_process(max_handle)


## Launch and Benchmark vLLM

This uses `vllm serve` with the same model and then benchmarks it with `max benchmark --backend vllm`.


In [ ]:
from gemma4_colab_benchmark_helper import (
    ensure_server_ready,
    start_logged_process,
    tail_log,
    warmup_chat,
)

server_env = os.environ.copy()
if HF_TOKEN:
    server_env["HF_TOKEN"] = HF_TOKEN
server_env["HF_HOME"] = os.environ["HF_HOME"]
server_env["VIRTUAL_ENV"] = str(VLLM_ENV)
server_env["PATH"] = f"{VLLM_ENV / 'bin'}:{server_env['PATH']}"
server_env["PYTHONNOUSERSITE"] = "1"
server_env["PYTHONPATH"] = VLLM_SITE_PACKAGES

vllm_handle = start_logged_process(
    "vllm_server",
    [
        VLLM_BIN,
        "serve",
        MODEL,
        "--host",
        "127.0.0.1",
        "--port",
        str(VLLM_PORT),
        "--max-model-len",
        str(MAX_MODEL_LEN),
        "--gpu-memory-utilization",
        str(GPU_MEMORY_UTILIZATION),
        "--limit-mm-per-prompt",
        "image=0,audio=0",
    ],
    log_dir=LOG_DIR,
    env=server_env,
)

ensure_server_ready(f"http://127.0.0.1:{VLLM_PORT}", timeout_s=1800)
warmup_chat(f"http://127.0.0.1:{VLLM_PORT}", MODEL)
print(tail_log(vllm_handle.log_path, lines=40))


In [ ]:
from gemma4_colab_benchmark_helper import load_benchmark_result, run_benchmark, summarize_result

vllm_result_path = run_benchmark(
    max_bin=MAX_BIN,
    backend="vllm",
    model=MODEL,
    port=VLLM_PORT,
    result_dir=RESULT_DIR,
    result_filename="vllm_random.json",
    dataset_name=BENCHMARK_DATASET,
    num_prompts=NUM_PROMPTS,
    request_rate=REQUEST_RATE,
    max_concurrency=MAX_CONCURRENCY,
    random_input_len=RANDOM_INPUT_LEN,
    random_output_len=RANDOM_OUTPUT_LEN,
    random_coefficient_of_variation=RANDOM_COV,
    tokenizer=MODEL,
    env=server_env,
)

vllm_result = load_benchmark_result(vllm_result_path)
vllm_summary = summarize_result(vllm_result, "vLLM")
vllm_summary


In [ ]:
from gemma4_colab_benchmark_helper import stop_process

stop_process(vllm_handle)


## Compare Results

The most important number for the blog claim is **output token throughput**. This cell also compares request throughput and latency metrics.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from gemma4_colab_benchmark_helper import percent_delta

comparison = pd.DataFrame([max_summary, vllm_summary]).set_index("label")
display(comparison)

output_delta_pct = percent_delta(
    comparison.loc["MAX", "output_throughput"],
    comparison.loc["vLLM", "output_throughput"],
)
request_delta_pct = percent_delta(
    comparison.loc["MAX", "request_throughput"],
    comparison.loc["vLLM", "request_throughput"],
)

if output_delta_pct is not None:
    print(f"MAX output throughput delta vs vLLM: {output_delta_pct:.2f}%")
if request_delta_pct is not None:
    print(f"MAX request throughput delta vs vLLM: {request_delta_pct:.2f}%")

plot_columns = [
    column
    for column in [
        "request_throughput",
        "output_throughput",
        "mean_ttft_ms",
        "mean_tpot_ms",
        "mean_itl_ms",
    ]
    if column in comparison.columns
]

comparison[plot_columns].plot(kind="bar", figsize=(10, 5), rot=0)
plt.title(f"{MODEL} on {gpu.name if gpu else 'unknown GPU'}")
plt.tight_layout()
plt.show()

RESULT_DIR.mkdir(parents=True, exist_ok=True)
summary_path = RESULT_DIR / "max_vs_vllm_summary.csv"
comparison.to_csv(summary_path)
print("Saved summary to", summary_path)
print("Raw benchmark JSON files are in", RESULT_DIR)


## How To Interpret What You See

- If your output throughput delta is positive, MAX was faster on **your** runtime and settings.
- If the notebook selected `allenai/OLMo-2-0425-1B-Instruct`, this run is only validating the notebook workflow and giving a directional MAX-vs-vLLM comparison on a small GPU.
- If your runtime was not a **B200**, do not treat the result as a direct reproduction of the Modular blog number.
- If the notebook had to fall back from `vllm==0.18.2rc1.dev7`, note that in your conclusions because the blog explicitly names that version.
- To get closer to the blog setup, rerun this notebook on a **B200** or at least an **80 GiB** GPU and use `google/gemma-4-26B-A4B-it`.
- This notebook intentionally uses a **single benchmark harness** across both engines to reduce client-side measurement bias.
